# Détection de fraude financière par Machine Learning
**Règles métier contre IA (Random Forest et XGBoost) sur PaySim**

PFE – Master 1 Finance, HESTIM – Imane Aghanim, encadrée par Mme Fatima Berrada.

Ce notebook reproduit les résultats du rapport, dans l'ordre :
1. Données et nettoyage
2. Découpage entraînement / test
3. Règle métier à 6 critères
4. Random Forest
5. XGBoost (par défaut, puis réglé)
6. Comparaison finale : précision, rappel, F1, coût
7. Robustesse : validation croisée et autres découpages
8. Analyses complémentaires : sans la variable « type », arbre simple, seuil de décision, filtre de règles, temps de détection

**Données :** PaySim (transactions de paiement mobile *simulées*), échantillon de 1 048 575 transactions. Le fichier n'est pas fourni : indiquer son chemin dans la cellule 1.

**Coûts (hypothèses, dirhams fictifs) :** 50 000 MAD par fraude manquée, 150 MAD par fausse alerte.

In [1]:
import time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, f1_score
from xgboost import XGBClassifier
from scipy.stats import binomtest
warnings.filterwarnings('ignore')

CHEMIN = 'paysim.csv'      # <-- à adapter : chemin du fichier PaySim
COUT_FRAUDE_MANQUEE = 50_000   # MAD (hypothèse)
COUT_FAUSSE_ALERTE  = 150      # MAD (hypothèse)
N_LIGNES = 1_048_575           # échantillon du rapport : premières lignes du fichier
SEED = 42

## 1. Données et nettoyage

In [2]:
# Détection automatique du séparateur (',' ou ';')
with open(CHEMIN, encoding='utf-8') as f:
    ligne = f.readline()
sep = ';' if ligne.count(';') > ligne.count(',') else ','

df = pd.read_csv(CHEMIN, sep=sep, nrows=N_LIGNES).drop_duplicates()
print('Transactions :', len(df), '| fraudes :', int(df['isFraud'].sum()),
      f"({df['isFraud'].mean()*100:.2f} %)")
print('Valeurs manquantes :', int(df.isna().sum().sum()))
df.head()

Transactions : 1048575 | fraudes : 1142 (0.11 %)
Valeurs manquantes : 0


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [3]:
# Nettoyage : on retire les numéros de compte, le type d'opération devient des colonnes 0/1
df = df.drop(columns=['nameOrig', 'nameDest'])
df = pd.get_dummies(df, columns=['type'], drop_first=True)
X = df.drop(columns='isFraud')
y = df['isFraud']
print(list(X.columns))

['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


## 2. Découpage entraînement / test
70 % pour entraîner, 30 % pour tester, avec la même part de fraudes des deux côtés (stratifié).

In [4]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
print('Entraînement :', len(Xtr), 'dont fraudes', int(ytr.sum()))
print('Test         :', len(Xte), 'dont fraudes', int(yte.sum()))

Entraînement : 734002 dont fraudes 799
Test         : 314573 dont fraudes 343


## 3. Mesures et coût
- Précision = VP ÷ (VP + FP) · Rappel = VP ÷ (VP + FN) · F1 = 2 × P × R ÷ (P + R)
- Coût = fraudes manquées × 50 000 + fausses alertes × 150

In [5]:
def bilan(nom, yt, yp):
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    return {'Méthode': nom, 'Manquées (FN)': int(fn), 'Fausses alertes (FP)': int(fp), 'Trouvées (VP)': int(tp),
            'Précision': float(round(tp/(tp+fp), 3)) if tp+fp else 0.0, 'Rappel': float(round(tp/(tp+fn), 3)),
            'F1': float(round(2*tp/(2*tp+fp+fn), 3)), 'Coût (MAD)': int(fn*COUT_FRAUDE_MANQUEE + fp*COUT_FAUSSE_ALERTE)}

# Exemple de calcul à la main : XGBoost réglé = 48 manquées, 28 fausses alertes
print(48*COUT_FRAUDE_MANQUEE + 28*COUT_FAUSSE_ALERTE)   # 2 404 200

2404200


## 4. Règle métier à 6 critères
Une transaction est signalée si **au moins k critères sur 6** sont réunis : montant > 200 000, compte émetteur vidé, type TRANSFER ou CASH_OUT, destinataire à solde 0, solde du destinataire non crédité du montant, opération de nuit (0 h – 6 h).

Le seuil k est choisi sur l'**entraînement** (coût minimal), puis évalué sur le test.

In [6]:
def criteres(D):
    risque = (D['type_TRANSFER'] == 1) | (D['type_CASH_OUT'] == 1)
    return pd.DataFrame({
        'montant_eleve': D['amount'] > 200_000,
        'compte_vide': (D['newbalanceOrig'] == 0) & (D['oldbalanceOrg'] > 0),
        'type_a_risque': risque,
        'destinataire_vide': D['oldbalanceDest'] == 0,
        'solde_dest_non_credite': (D['oldbalanceDest'] + D['amount'] - D['newbalanceDest']).abs() > 1,
        'nuit': (D['step'] % 24) < 6}).astype(int)

Ctr, Cte = criteres(Xtr), criteres(Xte)
lignes = []
for k in range(1, 7):
    b = bilan(f'Règle : au moins {k} critères', yte, (Cte.sum(axis=1) >= k).astype(int))
    b['Coût sur entraînement'] = bilan('', ytr, (Ctr.sum(axis=1) >= k).astype(int))['Coût (MAD)']
    lignes.append(b)
tab_k = pd.DataFrame(lignes)
K = int(tab_k['Coût sur entraînement'].idxmin()) + 1
print('k retenu (coût minimal sur l\'entraînement) :', K)
tab_k

k retenu (coût minimal sur l'entraînement) : 4


,Méthode,Manquées (FN),Fausses alertes (FP),Trouvées (VP),Précision,Rappel,F1,Coût (MAD),Coût sur entraînement
0,Règle : au moins 1 critères,0,313004,343,0.001,1.000,0.002,46950600,109564650
1,Règle : au moins 2 critères,1,241236,342,0.001,0.997,0.003,36235400,84629600
2,Règle : au moins 3 critères,37,75850,306,0.004,0.892,0.008,13227500,30519200
3,Règle : au moins 4 critères,111,18914,232,0.012,0.676,0.024,8387100,19421650
4,Règle : au moins 5 critères,221,2405,122,0.048,0.356,0.085,11410750,26788800
5,Règle : au moins 6 critères,319,3,24,0.889,0.070,0.130,15950450,37650900


In [7]:
regle = (Cte.sum(axis=1) >= K).astype(int).values
res_regle = bilan(f'Règle métier (≥ {K} critères sur 6)', yte, regle)
res_regle

{'Méthode': 'Règle métier (≥ 4 critères sur 6)',
 'Manquées (FN)': 111,
 'Fausses alertes (FP)': 18914,
 'Trouvées (VP)': 232,
 'Précision': 0.012,
 'Rappel': 0.676,
 'F1': 0.024,
 'Coût (MAD)': 8387100}

## 5. Random Forest
Bagging : 100 arbres indépendants, classes pondérées pour tenir compte de la rareté des fraudes. Non réglé.

In [8]:
t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=SEED, class_weight='balanced', n_jobs=-1).fit(Xtr, ytr)
p_rf = rf.predict_proba(Xte)[:, 1]
y_rf = (p_rf >= 0.5).astype(int)
res_rf = bilan('Random Forest', yte, y_rf)
print(round(time.time()-t0), 's'); res_rf

27 s


{'Méthode': 'Random Forest',
 'Manquées (FN)': 70,
 'Fausses alertes (FP)': 12,
 'Trouvées (VP)': 273,
 'Précision': 0.958,
 'Rappel': 0.796,
 'F1': 0.869,
 'Coût (MAD)': 3501800}

## 6. XGBoost
Boosting : arbres construits un par un, chacun corrige les erreurs du précédent.
D'abord les paramètres par défaut (gardés pour comparaison), puis le réglage.

In [9]:
xgb_defaut = XGBClassifier(random_state=SEED, eval_metric='logloss', n_jobs=-1).fit(Xtr, ytr)
y_xgd = xgb_defaut.predict(Xte)
res_xgd = bilan('XGBoost par défaut', yte, y_xgd)
res_xgd

{'Méthode': 'XGBoost par défaut',
 'Manquées (FN)': 63,
 'Fausses alertes (FP)': 18,
 'Trouvées (VP)': 280,
 'Précision': 0.94,
 'Rappel': 0.816,
 'F1': 0.874,
 'Coût (MAD)': 3152700}

### Réglage de XGBoost
Huit combinaisons : profondeur 3 ou 6 × pondération de la fraude (`scale_pos_weight`) de 1, 10, 30 ou 918 (918 ≈ rapport transactions saines / fraudes).
Validation croisée à 3 plis, sur l'**entraînement uniquement**, critère : F1.

In [10]:
t0 = time.time()
ratio = (ytr == 0).sum() / (ytr == 1).sum()
print('rapport saines / fraudes :', round(ratio))
cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
essais = []
for spw in [1, 10, 30, round(ratio)]:
    for prof in [3, 6]:
        m = XGBClassifier(n_estimators=100, max_depth=prof, scale_pos_weight=spw, random_state=SEED,
                          eval_metric='logloss', tree_method='hist', n_jobs=-1)
        s = cross_val_score(m, Xtr, ytr, cv=cv3, scoring='f1')
        essais.append({'scale_pos_weight': spw, 'max_depth': prof, 'F1 moyen': round(s.mean(), 3), 'écart-type': round(s.std(), 3)})
essais = pd.DataFrame(essais).sort_values('F1 moyen', ascending=False)
print(round(time.time()-t0), 's')
essais

rapport saines / fraudes : 918


48 s


,scale_pos_weight,max_depth,F1 moyen,écart-type
2,10,3,0.897,0.015
3,10,6,0.886,0.014
5,30,6,0.885,0.017
0,1,3,0.882,0.014
7,918,6,0.843,0.012
4,30,3,0.818,0.022
1,1,6,0.802,0.075
6,918,3,0.300,0.030


In [11]:
# Modèle retenu : profondeur 3, pondération 10, 100 arbres
PARAMS_XGB = dict(n_estimators=100, max_depth=3, scale_pos_weight=10, random_state=SEED,
                  eval_metric='logloss', tree_method='hist', n_jobs=-1)
xgb = XGBClassifier(**PARAMS_XGB).fit(Xtr, ytr)
p_xgb = xgb.predict_proba(Xte)[:, 1]
y_xgb = (p_xgb >= 0.5).astype(int)
res_xgb = bilan('XGBoost réglé', yte, y_xgb)
res_xgb

{'Méthode': 'XGBoost réglé',
 'Manquées (FN)': 48,
 'Fausses alertes (FP)': 28,
 'Trouvées (VP)': 295,
 'Précision': 0.913,
 'Rappel': 0.86,
 'F1': 0.886,
 'Coût (MAD)': 2404200}

## 7. Comparaison finale (jeu de test)

In [12]:
final = pd.DataFrame([res_regle, res_rf, res_xgd, res_xgb])
final

,Méthode,Manquées (FN),Fausses alertes (FP),Trouvées (VP),Précision,Rappel,F1,Coût (MAD)
0,Règle métier (≥ 4 critères sur 6),111,18914,232,0.012,0.676,0.024,8387100
1,Random Forest,70,12,273,0.958,0.796,0.869,3501800
2,XGBoost par défaut,63,18,280,0.940,0.816,0.874,3152700
3,XGBoost réglé,48,28,295,0.913,0.860,0.886,2404200


In [13]:
# Résumé en une phrase
cr, cx = res_rf['Coût (MAD)'], res_xgb['Coût (MAD)']
print(f"Coût : règle {res_regle['Coût (MAD)']:,} | RF {cr:,} | XGBoost réglé {cx:,} MAD")
print(f"L'IA coûte {res_regle['Coût (MAD)']/cr:.1f} à {res_regle['Coût (MAD)']/cx:.1f} fois moins que la règle")
print(f"XGBoost réglé est {100*(1-cx/cr):.0f} % moins cher que le Random Forest")

Coût : règle 8,387,100 | RF 3,501,800 | XGBoost réglé 2,404,200 MAD
L'IA coûte 2.4 à 3.5 fois moins que la règle
XGBoost réglé est 31 % moins cher que le Random Forest


## 8. Robustesse
### 8.1 Validation croisée à 5 plis (sur l'entraînement)
Chaque pli sert une fois de test : on obtient 5 scores F1. Un modèle stable garde des scores proches.

In [14]:
t0 = time.time()
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
modeles = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=SEED, class_weight='balanced', n_jobs=-1),
    'XGBoost par défaut': XGBClassifier(random_state=SEED, eval_metric='logloss', n_jobs=-1),
    'XGBoost réglé': XGBClassifier(**PARAMS_XGB)}
lignes = []
for nom, m in modeles.items():
    s = cross_val_score(m, Xtr, ytr, cv=cv5, scoring='f1')
    lignes.append({'Modèle': nom, 'F1 par pli': ' / '.join(f'{v:.3f}' for v in s),
                   'min': round(s.min(), 3), 'max': round(s.max(), 3), 'moyenne': round(s.mean(), 3), 'écart-type': round(s.std(), 3)})
print(round(time.time()-t0), 's')
pd.DataFrame(lignes)

126 s


,Modèle,F1 par pli,min,max,moyenne,écart-type
0,Random Forest,0.878 / 0.842 / 0.893 / 0.880 / 0.893,0.842,0.893,0.877,0.019
1,XGBoost par défaut,0.799 / 0.867 / 0.856 / 0.822 / 0.396,0.396,0.867,0.748,0.178
2,XGBoost réglé,0.890 / 0.871 / 0.889 / 0.883 / 0.914,0.871,0.914,0.889,0.014


### 8.2 Cinq autres découpages train/test
On change le hasard du découpage (graines 0 à 4) et on compare le coût de XGBoost réglé et du Random Forest.

In [15]:
t0 = time.time()
lignes = []
for s in range(5):
    A, B, ya, yb = train_test_split(X, y, test_size=0.3, random_state=s, stratify=y)
    r = RandomForestClassifier(n_estimators=100, random_state=SEED, class_weight='balanced', n_jobs=-1).fit(A, ya)
    x = XGBClassifier(**PARAMS_XGB).fit(A, ya)
    br, bx = bilan('RF', yb, r.predict(B)), bilan('XGB', yb, x.predict(B))
    lignes.append({'graine': s, 'coût RF': br['Coût (MAD)'], 'coût XGBoost réglé': bx['Coût (MAD)'], 'F1 RF': br['F1'], 'F1 XGBoost réglé': bx['F1']})
R = pd.DataFrame(lignes)
print(round(time.time()-t0), 's')
print('XGBoost réglé moins cher dans', int((R['coût XGBoost réglé'] < R['coût RF']).sum()), '/ 5 découpages')
print(R[['coût RF', 'coût XGBoost réglé', 'F1 RF', 'F1 XGBoost réglé']].mean().round(3).to_dict())
R

156 s
XGBoost réglé moins cher dans 5 / 5 découpages
{'coût RF': 3061530.0, 'coût XGBoost réglé': 2195130.0, 'F1 RF': 0.888, 'F1 XGBoost réglé': 0.885}


,graine,coût RF,coût XGBoost réglé,F1 RF,F1 XGBoost réglé
0,0,3201500,2106150,0.883,0.879
1,1,2951950,2055100,0.888,0.890
2,2,3051200,2304500,0.891,0.887
3,3,3102400,2306000,0.878,0.874
4,4,3000600,2203900,0.898,0.895


### 8.3 Test de McNemar (sur les fraudes)
Parmi les 343 fraudes, combien trouvées par un seul des deux modèles ? Test exact (binomial) sur les désaccords.

In [16]:
fr = yte.values == 1
seul_xgb = int(((y_xgb[fr] == 1) & (y_rf[fr] == 0)).sum())
seul_rf  = int(((y_xgb[fr] == 0) & (y_rf[fr] == 1)).sum())
p = binomtest(seul_xgb, seul_xgb + seul_rf, 0.5).pvalue if seul_xgb + seul_rf else 1.0
print('Fraudes trouvées seulement par XGBoost réglé :', seul_xgb)
print('Fraudes trouvées seulement par le Random Forest :', seul_rf)
print('p-value :', p)

Fraudes trouvées seulement par XGBoost réglé : 22
Fraudes trouvées seulement par le Random Forest : 0
p-value : 4.76837158203125e-07


## 9. Analyses complémentaires

### 9.1 Importance des variables

In [17]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
imp.round(3).to_frame('Importance (Random Forest)')

,Importance (Random Forest)
oldbalanceOrg,0.236
step,0.170
newbalanceOrig,0.166
amount,0.129
type_PAYMENT,0.074
oldbalanceDest,0.068
type_TRANSFER,0.068
newbalanceDest,0.057
type_CASH_OUT,0.033
type_DEBIT,0.000


### 9.2 Que se passe-t-il sans la variable « type » ?

In [18]:
cols = [c for c in Xtr.columns if not c.startswith('type_')]
rf_st = RandomForestClassifier(n_estimators=100, random_state=SEED, class_weight='balanced', n_jobs=-1).fit(Xtr[cols], ytr)
xg_st = XGBClassifier(**PARAMS_XGB).fit(Xtr[cols], ytr)
pd.DataFrame([res_rf, bilan('Random Forest sans « type »', yte, rf_st.predict(Xte[cols])),
              res_xgb, bilan('XGBoost réglé sans « type »', yte, xg_st.predict(Xte[cols]))])

,Méthode,Manquées (FN),Fausses alertes (FP),Trouvées (VP),Précision,Rappel,F1,Coût (MAD)
0,Random Forest,70,12,273,0.958,0.796,0.869,3501800
1,Random Forest sans « type »,102,13,241,0.949,0.703,0.807,5101950
2,XGBoost réglé,48,28,295,0.913,0.860,0.886,2404200
3,XGBoost réglé sans « type »,57,67,286,0.810,0.834,0.822,2860050


### 9.3 Un arbre de décision simple fait-il aussi bien ?

In [19]:
lignes = []
for prof in [2, 3, 4]:
    dt = DecisionTreeClassifier(max_depth=prof, class_weight='balanced', random_state=SEED).fit(Xtr, ytr)
    lignes.append(bilan(f'Arbre de profondeur {prof}', yte, dt.predict(Xte)))
pd.DataFrame(lignes)

,Méthode,Manquées (FN),Fausses alertes (FP),Trouvées (VP),Précision,Rappel,F1,Coût (MAD)
0,Arbre de profondeur 2,82,29731,261,0.009,0.761,0.017,8559650
1,Arbre de profondeur 3,65,6046,278,0.044,0.810,0.083,4156900
2,Arbre de profondeur 4,66,1933,277,0.125,0.808,0.217,3589950


### 9.4 Seuil de décision
Jusqu'ici le seuil est de 0,5. Ici on choisit le seuil qui minimise le coût sur l'**entraînement** (prédictions hors échantillon, 3 plis), puis on l'applique au test.

In [20]:
t0 = time.time()
cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
seuils = np.arange(0.05, 0.96, 0.05)
def cout(yt, yp):
    return bilan('', yt, yp)['Coût (MAD)']
lignes = []
for nom, mk, p_test in [('Random Forest', lambda: RandomForestClassifier(n_estimators=100, random_state=SEED, class_weight='balanced', n_jobs=-1), p_rf),
                        ('XGBoost réglé', lambda: XGBClassifier(**PARAMS_XGB), p_xgb)]:
    oof = cross_val_predict(mk(), Xtr, ytr, cv=cv3, method='predict_proba')[:, 1]
    couts = [cout(ytr, (oof >= t).astype(int)) for t in seuils]
    t_opt = float(seuils[int(np.argmin(couts))])
    lignes.append(bilan(f'{nom}, seuil {t_opt:.2f}', yte, (p_test >= t_opt).astype(int)))
print(round(time.time()-t0), 's')
pd.DataFrame(lignes)

61 s


,Méthode,Manquées (FN),Fausses alertes (FP),Trouvées (VP),Précision,Rappel,F1,Coût (MAD)
0,"Random Forest, seuil 0.05",25,626,318,0.337,0.927,0.494,1343900
1,"XGBoost réglé, seuil 0.05",12,1206,331,0.215,0.965,0.352,780900


### 9.5 Filtrer d'abord avec la règle, puis laisser décider le modèle ?
Alerte seulement si la règle (≥ k critères) **et** XGBoost réglé sont d'accord.

In [21]:
lignes = []
for k in [2, 3, 4]:
    ru = (Cte.sum(axis=1) >= k).values
    lignes.append(bilan(f'Règle (≥ {k}) ET XGBoost réglé', yte, (ru & (y_xgb == 1)).astype(int)))
lignes.append(res_xgb)
pd.DataFrame(lignes)

,Méthode,Manquées (FN),Fausses alertes (FP),Trouvées (VP),Précision,Rappel,F1,Coût (MAD)
0,Règle (≥ 2) ET XGBoost réglé,48,28,295,0.913,0.860,0.886,2404200
1,Règle (≥ 3) ET XGBoost réglé,69,25,274,0.916,0.799,0.854,3453750
2,Règle (≥ 4) ET XGBoost réglé,121,12,222,0.949,0.647,0.769,6051800
3,XGBoost réglé,48,28,295,0.913,0.860,0.886,2404200


### 9.6 Temps de détection

In [22]:
def chrono(f):
    t = time.time(); f(); return time.time() - t
def une(f, n=200):
    t = time.time()
    for _ in range(n): f()
    return (time.time() - t) / n * 1000
un = Xte.iloc[[0]]
lignes = []
for nom, lot, seule in [
    ('Règle métier', lambda: (criteres(Xte).sum(axis=1) >= K), lambda: (criteres(un).sum(axis=1) >= K)),
    ('Random Forest', lambda: rf.predict(Xte), lambda: rf.predict(un)),
    ('XGBoost réglé', lambda: xgb.predict(Xte), lambda: xgb.predict(un))]:
    t = chrono(lot)
    lignes.append({'Méthode': nom, 'Test entier (s)': round(t, 2), 'µs par transaction (lot)': round(t/len(Xte)*1e6, 1),
                   '1 transaction isolée (ms)': round(une(seule), 1)})
pd.DataFrame(lignes)

,Méthode,Test entier (s),µs par transaction (lot),1 transaction isolée (ms)
0,Règle métier,0.03,0.1,1.4
1,Random Forest,0.69,2.2,24.2
2,XGBoost réglé,0.15,0.5,1.7


## Limites (à garder en tête)
- Données **simulées** : la fraude est très structurée par type d'opération, les signaux sont probablement plus nets que dans une vraie banque.
- Coûts = **hypothèses** (dirhams fictifs, sans coût de mise en place). Les montants exacts sont fictifs ; le classement entre méthodes est ce qui compte.
- La règle métier a été construite pour l'étude ; ce n'est pas une règle bancaire réelle.
- Random Forest non réglé ; XGBoost réglé sur 8 combinaisons seulement.
- Seuil de décision à 0,5 pour les résultats principaux : un seuil plus bas réduit le coût mais fait chuter le F1.